In [1]:
import requests
import pandas as pd
import json

# Base API URL (assuming state ID 1..7 for all provinces)
base_url = "https://keyvalue.hamropatro.com/kv/get/major-election-2082-states-{}-result"

all_rows = []

# loop through all states (Koshi=1, Madhesh=2, Bagmati=3, Gandaki=4, Lumbini=5, Karnali=6, Sudurpashchim=7)
for state_id in range(1, 8):
    url = base_url.format(state_id)
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"})
    data = response.json()
    
    value = data.get("list")[0].get("value")
    if not value:
        print(f"⚠️ No data for state {state_id}")
        continue
    
    state_data = json.loads(value)
    
    for area in state_data.get("stateResults", []):
        area_name = area.get("areaNameEnglish")
        district_name = area.get("districtEnglishName")
        state_name = area.get("stateName")  # already English
        total_counted_votes = area.get("totalCountedVotes")
        total_cast_votes = area.get("totalCastVotes")
        result_status = area.get("electionResultStatus")
        
        for candidate in area.get("candidateResults", []):
            all_rows.append({
                "State": state_name,
                "District": district_name,
                "Area": area_name,
                "Candidate Name": candidate.get("englishName"),
                "Party": candidate.get("partyName"),
                "Votes": candidate.get("votes"),
                "Winner": candidate.get("winner"),
                "Featured": candidate.get("featured"),
                "Result Final": candidate.get("resultFinal"),
                "Image URL": candidate.get("imageUrl"),
                "Slug": candidate.get("slug"),
                "Total Counted Votes": total_counted_votes,
                "Total Cast Votes": total_cast_votes,
                "Result Status": result_status
            })

# Save to DataFrame and CSV
df = pd.DataFrame(all_rows)
df.to_csv("nepal_election_results_all_states.csv", index=False)
print(f"✅ CSV saved with {len(df)} rows for all states")

✅ CSV saved with 495 rows for all states


In [2]:
df

,State,District,Area,Candidate Name,Party,Votes,Winner,Featured,Result Final,Image URL,Slug,Total Counted Votes,Total Cast Votes,Result Status
0,Koshi Province,Taplejung,Area 1,Kshitij Thebe,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी ले...,13962,True,False,False,https://file-r2.hamropatro.com/election-images...,kshitij-thebe-a1,42897,0,RESULT_ANNOUNCED
1,Koshi Province,Taplejung,Area 1,Gajendra Prasad Tumyang Limbu,नेपाली काँग्रेस,11711,False,False,False,https://file-r2.hamropatro.com/election-images...,gajendra-prasad-tumyang-limbu-a1,42897,0,RESULT_ANNOUNCED
2,Koshi Province,Taplejung,Area 1,Khel Prasad Budakshetri,नेपाली कम्युनिष्ट पार्टी,6775,False,False,False,https://file-r2.hamropatro.com/election-images...,khel-prasad-budakshetri-a1,42897,0,RESULT_ANNOUNCED
3,Koshi Province,Panchthar,Area 1,Narendra Kumar Kerung,नेपाली काँग्रेस,17233,True,False,False,https://file-r2.hamropatro.com/election-images...,narendra-kumar-kerung-a2,66921,0,RESULT_ANNOUNCED
4,Koshi Province,Panchthar,Area 1,Hasta Raj Sharma,श्रम संस्कृति पार्टी,14734,False,False,False,https://file-r2.hamropatro.com/election-images...,hasta-raj-sharma-a2,66921,0,RESULT_ANNOUNCED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,Sudurpashchim Province,Kanchanpur,Area 2,Narayan Prakash Saud,नेपाली काँग्रेस,10827,False,False,False,https://file-r2.hamropatro.com/election-images...,narayan-prakash-saud-a161,61117,64881,RESULT_ANNOUNCED
491,Sudurpashchim Province,Kanchanpur,Area 2,Bachan Bahadur Singh,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी ले...,6418,False,False,False,https://file-r2.hamropatro.com/election-images...,bachan-bahadur-singh-a161,61117,64881,RESULT_ANNOUNCED
492,Sudurpashchim Province,Kanchanpur,Area 3,Gyanendra Singh Mahata,राष्ट्रिय स्वतन्त्र पार्टी,27118,True,False,False,https://file-r2.hamropatro.com/election-images...,gyanendra-singh-mahata-a162,53546,0,RESULT_ANNOUNCED
493,Sudurpashchim Province,Kanchanpur,Area 3,Hari Prasad Bohra,नेपाली काँग्रेस,9443,False,False,False,https://file-r2.hamropatro.com/election-images...,hari-prasad-bohra-a162,53546,0,RESULT_ANNOUNCED


In [5]:
df['Party'].value_counts()

Party
नेपाली काँग्रेस                                           148
राष्ट्रिय स्वतन्त्र पार्टी                                142
नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी लेनिनवादी)    114
नेपाली कम्युनिष्ट पार्टी                                   48
श्रम संस्कृति पार्टी                                       13
जनता समाजवादी पार्टी, नेपाल                                11
राष्ट्रिय प्रजातन्त्र पार्टी                                5
स्वतन्त्र                                                   3
जनमत पार्टी                                                 2
उज्यालो नेपाल पार्टी                                        2
स्वाभिमान पार्टी                                            1
नेपाल संघीय समाजवादी पार्टी(एकल चुनाव चिन्ह)                1
आम जनता पार्टी(एकल चुनाव चिन्ह)                             1
नेपाल मजदुर किसान पार्टी                                    1
मंगोल नेशनल अर्गनाइजेसन                                     1
प्रगतिशील लोकतान्त्रिक पार्टी                               1
ना

In [6]:
party_map = {
    "नेपाली काँग्रेस": "Nepali Congress",
    "राष्ट्रिय स्वतन्त्र पार्टी": "Rastriya Swatantra Party",
    "नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी लेनिनवादी)": "Nepal Communist Party (UML)",
    "नेपाली कम्युनिष्ट पार्टी": "Nepali Communist Party",
    "श्रम संस्कृति पार्टी": "Shram Sanskriti Party",
    "जनता समाजवादी पार्टी, नेपाल": "Janata Samajwadi Party, Nepal",
    "राष्ट्रिय प्रजातन्त्र पार्टी": "Rastriya Prajatantra Party",
    "स्वतन्त्र": "Independent",
    "जनमत पार्टी": "Janamat Party",
    "उज्यालो नेपाल पार्टी": "Ujyalo Nepal Party",
    "स्वाभिमान पार्टी": "Swabhiman Party",
    "नेपाल संघीय समाजवादी पार्टी(एकल चुनाव चिन्ह)": "Nepal Federal Socialist Party (Single Symbol)",
    "आम जनता पार्टी(एकल चुनाव चिन्ह)": "Aam Janata Party (Single Symbol)",
    "नेपाल मजदुर किसान पार्टी": "Nepal Workers and Peasants Party",
    "मंगोल नेशनल अर्गनाइजेसन": "Mongol National Organization",
    "प्रगतिशील लोकतान्त्रिक पार्टी": "Pragatisheel Loktantrik Party",
    "नागरिक उन्मुक्ति पार्टी, नेपाल(एकल चुनाव चिन्ह)": "Citizen Liberation Party, Nepal (Single Symbol)"
}

# Example of applying mapping in pandas
df['Party_English'] = df['Party'].map(party_map)

In [7]:
df['Party_English']

0      Nepal Communist Party (UML)
1                  Nepali Congress
2           Nepali Communist Party
3                  Nepali Congress
4            Shram Sanskriti Party
                  ...             
490                Nepali Congress
491    Nepal Communist Party (UML)
492       Rastriya Swatantra Party
493                Nepali Congress
494    Nepal Communist Party (UML)
Name: Party_English, Length: 495, dtype: object

In [8]:
df

,State,District,Area,Candidate Name,Party,Votes,Winner,Featured,Result Final,Image URL,Slug,Total Counted Votes,Total Cast Votes,Result Status,Party_English
0,Koshi Province,Taplejung,Area 1,Kshitij Thebe,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी ले...,13962,True,False,False,https://file-r2.hamropatro.com/election-images...,kshitij-thebe-a1,42897,0,RESULT_ANNOUNCED,Nepal Communist Party (UML)
1,Koshi Province,Taplejung,Area 1,Gajendra Prasad Tumyang Limbu,नेपाली काँग्रेस,11711,False,False,False,https://file-r2.hamropatro.com/election-images...,gajendra-prasad-tumyang-limbu-a1,42897,0,RESULT_ANNOUNCED,Nepali Congress
2,Koshi Province,Taplejung,Area 1,Khel Prasad Budakshetri,नेपाली कम्युनिष्ट पार्टी,6775,False,False,False,https://file-r2.hamropatro.com/election-images...,khel-prasad-budakshetri-a1,42897,0,RESULT_ANNOUNCED,Nepali Communist Party
3,Koshi Province,Panchthar,Area 1,Narendra Kumar Kerung,नेपाली काँग्रेस,17233,True,False,False,https://file-r2.hamropatro.com/election-images...,narendra-kumar-kerung-a2,66921,0,RESULT_ANNOUNCED,Nepali Congress
4,Koshi Province,Panchthar,Area 1,Hasta Raj Sharma,श्रम संस्कृति पार्टी,14734,False,False,False,https://file-r2.hamropatro.com/election-images...,hasta-raj-sharma-a2,66921,0,RESULT_ANNOUNCED,Shram Sanskriti Party
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,Sudurpashchim Province,Kanchanpur,Area 2,Narayan Prakash Saud,नेपाली काँग्रेस,10827,False,False,False,https://file-r2.hamropatro.com/election-images...,narayan-prakash-saud-a161,61117,64881,RESULT_ANNOUNCED,Nepali Congress
491,Sudurpashchim Province,Kanchanpur,Area 2,Bachan Bahadur Singh,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी ले...,6418,False,False,False,https://file-r2.hamropatro.com/election-images...,bachan-bahadur-singh-a161,61117,64881,RESULT_ANNOUNCED,Nepal Communist Party (UML)
492,Sudurpashchim Province,Kanchanpur,Area 3,Gyanendra Singh Mahata,राष्ट्रिय स्वतन्त्र पार्टी,27118,True,False,False,https://file-r2.hamropatro.com/election-images...,gyanendra-singh-mahata-a162,53546,0,RESULT_ANNOUNCED,Rastriya Swatantra Party
493,Sudurpashchim Province,Kanchanpur,Area 3,Hari Prasad Bohra,नेपाली काँग्रेस,9443,False,False,False,https://file-r2.hamropatro.com/election-images...,hari-prasad-bohra-a162,53546,0,RESULT_ANNOUNCED,Nepali Congress
